In [1]:
import sys
import importlib

print("="*60)
print(f"📍 Môi trường Python đang chạy: {sys.executable}")
print(f"🐍 Phiên bản Python: {sys.version.split()[0]}")
print("="*60)

# Danh sách các thư viện cần kiểm tra
libraries = {
    "torch": "PyTorch (Chạy model AI)",
    "llama_cpp": "Llama-cpp-python (Chạy Qwen)",
    "langchain": "LangChain (Khung RAG)",
    "langchain_community": "LangChain Community",
    "langchain_huggingface": "LangChain HuggingFace",
    "sentence_transformers": "Sentence Transformers (Embedding)",
    "faiss": "FAISS (Vector Database)",
    "tqdm": "TQDM (Thanh tiến trình)"
}

missing_libs = []

for lib, description in libraries.items():
    try:
        # Cố gắng import thư viện
        module = importlib.import_module(lib)
        # Lấy phiên bản (nếu có)
        version = getattr(module, "__version__", "Không rõ phiên bản")
        print(f"✅ {lib.ljust(22)} | {description.ljust(30)} | Version: {version}")
    except ImportError:
        print(f"❌ {lib.ljust(22)} | {description.ljust(30)} | CHƯA CÀI ĐẶT!")
        missing_libs.append(lib)

print("="*60)
if not missing_libs:
    print("🎉 TUYỆT VỜI! Tất cả các thư viện đã được cài đặt thành công.")
    print("Bạn đã sẵn sàng chạy code build FAISS và Qwen!")
else:
    print(f"⚠️ CẢNH BÁO: Bạn còn thiếu {len(missing_libs)} thư viện: {missing_libs}")
    print("Hãy chạy lệnh cài đặt lại các thư viện còn thiếu này (nhớ dùng đúng môi trường chatbot).")
print("="*60)

📍 Môi trường Python đang chạy: c:\Users\dinht\anaconda3\envs\chatbot\python.exe
🐍 Phiên bản Python: 3.11.16
✅ torch                  | PyTorch (Chạy model AI)        | Version: 2.14.1+cpu
✅ llama_cpp              | Llama-cpp-python (Chạy Qwen)   | Version: 0.3.36
✅ langchain              | LangChain (Khung RAG)          | Version: 1.4.6
✅ langchain_community    | LangChain Community            | Version: 0.4.2
✅ langchain_huggingface  | LangChain HuggingFace          | Version: 1.2.3


c:\Users\dinht\anaconda3\envs\chatbot\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ sentence_transformers  | Sentence Transformers (Embedding) | Version: 6.1.0
✅ faiss                  | FAISS (Vector Database)        | Version: 1.15.1
✅ tqdm                   | TQDM (Thanh tiến trình)        | Version: 4.70.1
🎉 TUYỆT VỜI! Tất cả các thư viện đã được cài đặt thành công.
Bạn đã sẵn sàng chạy code build FAISS và Qwen!


In [2]:
import os

INPUT_FILE = r"C:\Users\dinht\Downloads\Bo_Cai_Chatbot_Luat\chunks.jsonl"
FAISS_DIR = r"C:\Users\dinht\Downloads\Bo_Cai_Chatbot_Luat\faiss_index_local"
BGE_M3_PATH = r"C:\Users\dinht\Downloads\Bo_Cai_Chatbot_Luat\models\bge-m3"
QWEN_MODEL_PATH = r"C:\Users\dinht\Downloads\Bo_Cai_Chatbot_Luat\qwen_model\qwen2.5-3b-instruct-q4_k_m.gguf"

print("✅ Đã nạp đường dẫn thành công!")

✅ Đã nạp đường dẫn thành công!


In [6]:
file_path = r"C:\Users\dinht\Downloads\Bo_Cai_Chatbot_Luat\chunks.jsonl"

print("⏳ Đang đếm số dòng...")
with open(file_path, 'rb') as f:
    lines = sum(1 for _ in f)
    
# Đã thêm dấu ngoặc đơn ở cuối dòng này
print(f"🔢 Tổng số chunks trong file là: {lines:,} dòng") 

⏳ Đang đếm số dòng...
🔢 Tổng số chunks trong file là: 455,578 dòng


In [7]:
import json
import hashlib
from tqdm.auto import tqdm

INPUT_FILE = r"C:\Users\dinht\Downloads\Bo_Cai_Chatbot_Luat\chunks.jsonl"
OUTPUT_FILE = r"C:\Users\dinht\Downloads\Bo_Cai_Chatbot_Luat\chunks_cleaned.jsonl"

INVALID_STATUS = ['hết hiệu lực toàn bộ', 'hết hiệu lực một phần', 'không còn phù hợp', 'đã bị thay thế', 'hết hiệu lực']
seen_hashes = set()
count_valid = 0

print("⏳ Đang làm sạch dữ liệu (lọc luật cũ & trùng lặp)...")
with open(INPUT_FILE, 'r', encoding='utf-8') as fin, open(OUTPUT_FILE, 'w', encoding='utf-8') as fout:
    for line in tqdm(fin, desc="Đang lọc"):
        data = json.loads(line)
        status = str(data.get('tinh_trang_hieu_luc', '')).lower()
        
        # 1. Bỏ qua luật hết hiệu lực
        if any(inv in status for inv in INVALID_STATUS):
            continue
            
        text = data.get('text', '').strip()
        if not text: continue
        
        # 2. Bỏ qua văn bản trùng lặp
        text_hash = hashlib.md5(text.encode('utf-8')).hexdigest()
        if text_hash in seen_hashes: continue
        seen_hashes.add(text_hash)
        
        fout.write(json.dumps(data, ensure_ascii=False) + '\n')
        count_valid += 1

print(f"\n✅ Đã lọc xong! Giữ lại {count_valid:,} chunks hợp lệ.")

⏳ Đang làm sạch dữ liệu (lọc luật cũ & trùng lặp)...


Đang lọc: 455578it [00:22, 20556.73it/s]


✅ Đã lọc xong! Giữ lại 163,821 chunks hợp lệ.


In [10]:
%pip install rank_bm25

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import json
import time
from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document
from langchain_community.llms import LlamaCpp
from langchain_core.prompts import PromptTemplate

# Đường dẫn file đã làm sạch và mô hình Qwen
INPUT_FILE = r"C:\Users\dinht\Downloads\Bo_Cai_Chatbot_Luat\chunks_cleaned.jsonl"
QWEN_MODEL_PATH = r"C:\Users\dinht\Downloads\Bo_Cai_Chatbot_Luat\qwen_model\qwen2.5-3b-instruct-q4_k_m.gguf"

print("⏳ Đang nạp dữ liệu vào BM25 (Tìm kiếm từ khóa)...")
docs = []
with open(INPUT_FILE, 'r', encoding='utf-8') as f:
    for line in f:
        data = json.loads(line)
        docs.append(Document(
            page_content=data['text'], 
            metadata={"title": data.get("title")}
        ))

# Khởi tạo BM25 (chạy trong 1-2 giây)
retriever = BM25Retriever.from_documents(docs)
retriever.k = 3 # Lấy 3 đoạn văn bản khớp từ khóa nhất

print("⏳ Đang tải Qwen 2.5 3B...")
llm = LlamaCpp(
    model_path=QWEN_MODEL_PATH, 
    temperature=0.0, 
    max_tokens=512, 
    n_ctx=4096, 
    verbose=False
)

template = """<|im_start|>system
Bạn là trợ lý pháp lý. Chỉ trả lời dựa vào Ngữ cảnh được cung cấp. Nếu không có thông tin, hãy nói "Không tìm thấy thông tin."<|im_end|>
<|im_start|>user
[NGỮ CẢNH]
{context}

[CÂU HỎI]
{question}<|im_end|>
<|im_start|>assistant
"""
prompt = PromptTemplate.from_template(template)

def hoi_chatbot(question):
    print(f"\n❓ Câu hỏi: {question}")
    print("🤖 Đang tìm kiếm từ khóa và trả lời...")
    start_time = time.time()
    
    # Tìm kiếm bằng BM25
    docs = retriever.invoke(question)
    context = "\n\n".join([d.page_content for d in docs])
    
    # Gọi Qwen
    response = llm.invoke(prompt.format(context=context, question=question))
    answer = response.split("<|im_start|>assistant")[-1].strip()
    
    print(f"✅ Trả lời:\n{answer}")
    print(f"\n⏱️ Thời gian: {time.time() - start_time:.2f} giây")
    print("="*60)

# Test ngay câu hỏi của bạn
hoi_chatbot("Tôi có bằng lái B2 thì có được chạy xe tải không, và chở được tối đa bao nhiêu người?")

⏳ Đang nạp dữ liệu vào BM25 (Tìm kiếm từ khóa)...


In [ ]:
import json
import os
import time
import torch
import pickle
from langchain_community.retrievers import BM25Retriever
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.llms import LlamaCpp
from langchain_core.documents import Document

try:
    from langchain.retrievers import EnsembleRetriever, ContextualCompressionRetriever
    from langchain.retrievers.document_compressors import CrossEncoderReranker
except ImportError:
    from langchain_classic.retrievers import EnsembleRetriever, ContextualCompressionRetriever
    from langchain_classic.retrievers.document_compressors import CrossEncoderReranker

from langchain_community.cross_encoders import HuggingFaceCrossEncoder

# CẤU HÌNH ĐƯỜNG DẪN
CLEANED_FILE = r"C:\Users\dinht\Downloads\Bo_Cai_Chatbot_Luat\chunks_cleaned.jsonl"
FAISS_DIR = r"C:\Users\dinht\Downloads\Bo_Cai_Chatbot_Luat\faiss_index_local"
BM25_PICKLE = r"C:\Users\dinht\Downloads\Bo_Cai_Chatbot_Luat\bm25_retriever.pkl"
QWEN_MODEL_PATH = r"C:\Users\dinht\Downloads\Bo_Cai_Chatbot_Luat\qwen_model\qwen2.5-3b-instruct-q4_k_m.gguf"

print("⏳ Đang tải mô hình và thiết lập hệ thống (Vui lòng đợi)...")
device = "cuda" if torch.cuda.is_available() else "cpu"

# 1. LOAD BM25
if os.path.exists(BM25_PICKLE):
    print("⏳ Đang tải BM25...")
    with open(BM25_PICKLE, 'rb') as f:
        bm25_retriever = pickle.load(f)
else:
    print("⏳ Đang tạo file BM25...")
    docs = []
    with open(CLEANED_FILE, 'r', encoding='utf-8') as f:
        for line in f:
            data = json.loads(line)
            docs.append(Document(page_content=data['text'], metadata={"title": data.get("title")}))
    bm25_retriever = BM25Retriever.from_documents(docs)
    with open(BM25_PICKLE, 'wb') as f:
        pickle.dump(bm25_retriever, f)

# 2. KIỂM TRA FAISS & KẾT HỢP
if os.path.exists(FAISS_DIR):
    print("✅ Tìm thấy FAISS. Đang khởi tạo Hybrid Search (BM25 + Vector)...")
    embeddings = HuggingFaceEmbeddings(model_name="keepitreal/vietnamese-sbert", model_kwargs={'device': device})
    vectorstore = FAISS.load_local(FAISS_DIR, embeddings, allow_dangerous_deserialization=True)
    
    ensemble_retriever = EnsembleRetriever(
        retrievers=[bm25_retriever, vectorstore.as_retriever(search_kwargs={"k": 6})],
        weights=[0.5, 0.5]
    )
    cross_encoder = HuggingFaceCrossEncoder(model_name="BAAI/bge-reranker-base")
    compressor = CrossEncoderReranker(model=cross_encoder, top_n=3)
    retriever = ContextualCompressionRetriever(base_compressor=compressor, base_retriever=ensemble_retriever)
else:
    print("⚠️ KHÔNG tìm thấy FAISS. Chỉ sử dụng BM25 (Tìm kiếm từ khóa).")
    print("💡 Mẹo: Hãy chạy code trên Colab để tạo thư mục faiss_index_local và copy vào máy để dùng Hybrid Search.")
    retriever = bm25_retriever # Fallback to BM25 only

# 3. LOAD QWEN
print("⏳ Đang tải Qwen 3B...")
llm = LlamaCpp(
    model_path=QWEN_MODEL_PATH, 
    temperature=0.1, 
    max_tokens=512, 
    n_ctx=4096, 
    n_gpu_layers=-1 if device == "cuda" else 0, 
    verbose=False
)

# 4. CHẠY CLI
def chat_cli():
    print("\n" + "="*60)
    print("🎉 CHATBOT LUẬT ĐÃ SẴN SÀNG")
    print("Gõ 'exit' hoặc 'thoát' để dừng.")
    print("="*60)
    
    while True:
        question = input("\n👤 Bạn hỏi: ")
        if question.lower().strip() in ['exit', 'quit', 'thoát']: break
        if not question.strip(): continue
            
        start_time = time.time()
        docs = retriever.invoke(question)
        context = "\n\n".join([d.page_content for d in docs])
        
        template = """<|im_start|>system
Bạn là trợ lý pháp lý. Chỉ trả lời dựa vào Ngữ cảnh được cung cấp. Nếu không có thông tin, hãy nói "Không tìm thấy thông tin."<|im_end|>
<|im_start|>user
[NGỮ CẢNH]
{context}

[CÂU HỎI]
{question}<|im_end|>
<|im_start|>assistant
"""
        final_prompt = template.format(context=context, question=question)
        
        print("-" * 50)
        for chunk in llm.stream(final_prompt):
            print(chunk, end="", flush=True)
            
        print("\n" + "-" * 50)
        print(f"⏱️ Thời gian phản hồi: {time.time() - start_time:.2f} giây")
        print("="*60)

if __name__ == "__main__":
    chat_cli()

⏳ Đang tải mô hình và thiết lập hệ thống (Vui lòng đợi)...
⏳ Đang tạo file BM25...
⚠️ KHÔNG tìm thấy FAISS. Chỉ sử dụng BM25 (Tìm kiếm từ khóa).
💡 Mẹo: Hãy chạy code trên Colab để tạo thư mục faiss_index_local và copy vào máy để dùng Hybrid Search.
⏳ Đang tải Qwen 3B...

🎉 CHATBOT LUẬT ĐÃ SẴN SÀNG
Gõ 'exit' hoặc 'thoát' để dừng.
